In [ ]:
import torch
import numpy as np
from int_utils import NEURON_INTERVENTIONS_DIR, PatchTarget, Config
from int_utils import SourceActivations, run_experiment, EXAMPLES_PATH

In [4]:
def patch_neurons(cfg: Config,
                  base: str,
                  source: str,
                  source_activations: SourceActivations):

    src_acts = source_activations(cfg, source, cfg.modules)

    neuron_effects = []

    for module_idx in range(len(cfg.modules)):
        module = cfg.modules[module_idx]
        src_out = src_acts[module_idx].to(cfg.device)

        module_effects = []

        for start in range(0, cfg.n_embed, cfg.batch_size):
            end = min(start + cfg.batch_size, cfg.n_embed)
            batch_idxs = list(range(start, end))

            len_batch = len(batch_idxs)
            bases = [base] * len_batch

            with torch.no_grad(), cfg.model.trace(bases):
                base_out = module.output

                rows = torch.arange(len_batch, device=cfg.device)
                neurons = torch.arange(start, end, device=cfg.device)

                base_out[rows, 1, neurons] = src_out[start:end]

                logits = cfg.model.output.logits[:, -1, :]
                m = (logits[:, cfg.she_tk] - logits[:, cfg.he_tk]).save()

            module_effects.extend(m.tolist())

        neuron_effects.append(module_effects)

    return np.array(neuron_effects)

In [5]:
patch_target = PatchTarget(
    patch_fn=patch_neurons,
    output_dir=NEURON_INTERVENTIONS_DIR,
    effect_shape=(13, 768)
)


run_experiment(EXAMPLES_PATH, patch_target)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Device: cpu
Loaded 2873 processed IDs.
man_direct: 2873 already saved; 0 remaining
Loaded 2873 processed IDs.
man_indirect: 2873 already saved; 0 remaining
Loaded 2873 processed IDs.
woman_direct: 2873 already saved; 0 remaining
Loaded 2873 processed IDs.
woman_indirect: 2873 already saved; 0 remaining
